In [1]:
# examples/demo_lengkap_mortapy_dengan_momen_seleksi.py

import mortapy as mp
import math
import os # Pastikan os diimpor

# Fungsi helper
def tampilkan_hasil(deskripsi_awal: str, hasil_objek: mp.ActuarialResult, verifikasi_manual_str: str = ""):
    print(deskripsi_awal)
    hasil_objek.show() # Ini akan memanggil _repr_latex_ di Jupyter atau print Deskripsi di terminal
    if verifikasi_manual_str:
        print(verifikasi_manual_str)
    print("-" * 60)

# --- Definisi Parameter Umum ---
print("=" * 70)
print("        DEMO LENGKAP MORTAPY (TERMASUK MOMEN & DASAR SELEKSI)")
print("=" * 70)

usia_x = 65
suku_bunga_untuk_kalkulator = 0.05
gender_pilihan_tabel = 'wanita'
n_temporary = 10 # Untuk momen temporary
n_years_prob = 5 # Untuk _np_x dan _nq_x
defer_m_prob = 3
death_u_prob = 7
offset_s_fom = 0.5 # Untuk mu_{x+s} dan pdf_{x+s}

# Parameter untuk Asumsi
qx_konstan_val = 0.03
omega_dm_val = 100.0
alpha_beta_val = 1.5 
mu_cfm_val = 0.025
gompertz_params_val = [0.0001, 1.1] # B, c
makeham_params_val = [0.0002, 0.00008, 1.12] # A, B, c

print("\n--- Parameter Umum yang Digunakan dalam Demo ---")
print(f"Usia awal (x)         : {usia_x}")
print(f"Suku bunga (i)        : {suku_bunga_untuk_kalkulator:.2%}")
print(f"Gender (untuk tabel)  : {gender_pilihan_tabel.capitalize()}")
print(f"Periode temporary (n) : {n_temporary} tahun")
print(f"Periode probabilitas (n): {n_years_prob} tahun")
print(f"Periode tunda (m)       : {defer_m_prob} tahun")
print(f"Periode kematian (u)    : {death_u_prob} tahun")
print(f"Offset waktu (s)        : {offset_s_fom} tahun")
print("-" * 60)

# ==============================================================================
# BAGIAN 1: PERHITUNGAN BERBASIS TABEL MORTALITA (TMI DEFAULT)
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 1: PERHITUNGAN BERBASIS TABEL MORTALITA (TMI DEFAULT)")
print(" (Fitur Seleksi & Ultima belum didemokan dengan tabel khusus, menggunakan ultima)")
print("=" * 70)

try:
    tabel_default = mp.load_default_table()
    print(f"\nBerhasil memuat tabel default: {tabel_default}\n")
    print(f"--- Menggunakan parameter: Usia = {usia_x}, Gender = {gender_pilihan_tabel}, n_temporary = {n_temporary} ---")

    # --- 1.1 Fungsi Probabilitas Dasar ---
    print("\n--- 1.1 Fungsi Probabilitas Dasar (Tabel) ---")
    hasil_tpx_tabel = mp.survival_prob_table(
        age=usia_x, n_years=n_years_prob, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel
    )
    tampilkan_hasil(f"a. Probabilitas Hidup _{{{n_years_prob}}}p_{{{usia_x}}}:", hasil_tpx_tabel)

    hasil_tqx_tabel = mp.death_prob_table(
        age=usia_x, n_years=n_years_prob, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel
    )
    tampilkan_hasil(f"b. Probabilitas Kematian _{{{n_years_prob}}}q_{{{usia_x}}}:", hasil_tqx_tabel)

    hasil_deferred_tabel = mp.deferred_death_prob_table(
        age=usia_x, deferral_period=defer_m_prob, n_years_death=death_u_prob,
        interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel
    )
    tampilkan_hasil(f"c. Prob. Kematian Ditunda _{{{defer_m_prob}}}|_{{{death_u_prob}}}q_{{{usia_x}}}:", hasil_deferred_tabel)

    hasil_fom_cfm_tabel = mp.fom_table(
        age=usia_x, t_offset=offset_s_fom, interest_rate=suku_bunga_untuk_kalkulator,
        gender=gender_pilihan_tabel, assumption_fractional='cfm'
    )
    tampilkan_hasil(f"d. Force of Mortality μ_{{{usia_x}+{offset_s_fom}}} (Interpolasi CFM):", hasil_fom_cfm_tabel)

    hasil_pdf_cfm_tabel = mp.pdf_death_table(
        age=usia_x, t_period=offset_s_fom, interest_rate=suku_bunga_untuk_kalkulator,
        gender=gender_pilihan_tabel, assumption_fractional='cfm'
    )
    tampilkan_hasil(f"e. PDF Kematian f_X({usia_x}+{offset_s_fom}) (Interpolasi CFM):", hasil_pdf_cfm_tabel)

    # --- 1.2 NSP dan Anuitas ---
    print("\n--- 1.2 NSP dan Anuitas (Tabel) ---")
    hasil_nsp_wl_tabel = mp.nsp_wl_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel)
    tampilkan_hasil(f"a. NSP Whole Life A_{{{usia_x}}}:", hasil_nsp_wl_tabel)

    hasil_pv_ann_tabel = mp.pv_annuity_due_wl_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel)
    tampilkan_hasil(f"b. PV Anuitas Whole Life Due ä_{{{usia_x}}}:", hasil_pv_ann_tabel)

    # --- 1.3 Momen Curtate ---
    print("\n--- 1.3 Momen Curtate Future Lifetime (Tabel) ---")
    ex_wl_tabel = mp.ex_curtate_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel)
    tampilkan_hasil(f"a. Ekspektasi Curtate (e_{{{usia_x}}}, WL):", ex_wl_tabel)
    ex_temp_tabel = mp.ex_curtate_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary)
    tampilkan_hasil(f"b. Ekspektasi Curtate (e_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temp):", ex_temp_tabel)
    e_sq_wl_tabel = mp.e_sq_curtate_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel)
    tampilkan_hasil(f"c. Momen Kedua Curtate (E[K_{{{usia_x}}}^2], WL):", e_sq_wl_tabel)
    var_k_wl_tabel = mp.var_k_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel)
    tampilkan_hasil(f"d. Variansi Curtate (Var[K_{{{usia_x}}}], WL):", var_k_wl_tabel)

    # --- 1.4 Momen Complete (dengan asumsi UDD untuk bagian fraksional) ---
    print("\n--- 1.4 Momen Complete Future Lifetime (Tabel, Aproksimasi UDD Fraksional) ---")
    ex_c_wl_tabel = mp.ex_complete_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, assumption_fractional='udd')
    tampilkan_hasil(f"a. Ekspektasi Complete (e_circ_{{{usia_x}}}, WL):", ex_c_wl_tabel)
    ex_c_temp_tabel = mp.ex_complete_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary, assumption_fractional='udd')
    tampilkan_hasil(f"b. Ekspektasi Complete (e_circ_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temp):", ex_c_temp_tabel)
    e_sq_c_wl_tabel = mp.e_sq_complete_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, assumption_fractional='udd')
    tampilkan_hasil(f"c. Momen Kedua Complete (E[T_{{{usia_x}}}^2], WL):", e_sq_c_wl_tabel)
    var_t_wl_tabel = mp.var_t_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, assumption_fractional='udd')
    tampilkan_hasil(f"d. Variansi Complete (Var[T_{{{usia_x}}}], WL):", var_t_wl_tabel)

except FileNotFoundError as e:
    print(f"\n[PERINGATAN] Gagal memuat tabel mortalita default: {e}")
    print("Bagian 1 demo (berbasis tabel) akan dilewati.")
except Exception as e_table:
    print(f"\n[ERROR] Terjadi kesalahan pada perhitungan berbasis tabel: {e_table}")
    # import traceback
    # traceback.print_exc()


# ==============================================================================
# BAGIAN 2: MOMEN BERBASIS ASUMSI DISTRIBUSI MURNI
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 2: MOMEN BERBASIS ASUMSI DISTRIBUSI MURNI")
print("=" * 70)

assumptions_to_test_all_funcs = [
    ('constant_qx', [qx_konstan_val], f"q_x konstan = {qx_konstan_val}"),
    ('de_moivre', [omega_dm_val], f"De Moivre (ω={int(omega_dm_val)})"),
    ('beta_distribution', [omega_dm_val, alpha_beta_val], f"Beta Dist. (ω={int(omega_dm_val)}, α={alpha_beta_val})"),
    ('constant_mu_cfm', [mu_cfm_val], f"CFM (μ={mu_cfm_val})"),
    ('gompertz', gompertz_params_val, f"Gompertz (B={gompertz_params_val[0]:.2e}, c={gompertz_params_val[1]})"),
    ('makeham', makeham_params_val, f"Makeham (A={makeham_params_val[0]:.2e}, B={makeham_params_val[1]:.2e}, c={makeham_params_val[2]})")
]

for i, (assumption_type, params, desc_short) in enumerate(assumptions_to_test_all_funcs):
    print(f"\n--- 2.{i+1} Menggunakan Asumsi: {desc_short} ---")
    print(f"   Parameter Umum: Usia = {usia_x}, Periode Float = {periode_t_float}, n_temporary = {n_temporary}")
    print(f"                   Deferral = {deferral_m}, Death Period = {death_period_u}, Offset FoM = {offset_s_fom}")
    print(f"   Parameter Asumsi: {params}\n")

    try:
        # --- Fungsi Probabilitas Dasar ---
        print(f"   --- Fungsi Probabilitas Dasar (Asumsi) ---")
        hasil_tpx_as = mp.survival_prob_assumption(age=usia_x, period=periode_t_float, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   1. Probabilitas Hidup _{{{periode_t_float}}}p_{{{usia_x}}}:", hasil_tpx_as)
        hasil_tqx_as = mp.death_prob_assumption(age=usia_x, period=periode_t_float, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   2. Probabilitas Kematian _{{{periode_t_float}}}q_{{{usia_x}}}:", hasil_tqx_as)
        hasil_deferred_as = mp.deferred_death_prob_assumption(age=usia_x, deferral_period=deferral_m, death_period=death_period_u, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   3. Prob. Kematian Ditunda _{{{deferral_m}}}|_{{{death_period_u}}}q_{{{usia_x}}}:", hasil_deferred_as)
        hasil_fom_as = mp.fom_assumption(age=usia_x, t_offset=offset_s_fom, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   4. Force of Mortality μ_{{{usia_x}+{offset_s_fom}}}:", hasil_fom_as)
        hasil_pdf_as = mp.pdf_death_assumption(age=usia_x, t_period=offset_s_fom, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   5. PDF Kematian f_X({usia_x}+{offset_s_fom}):", hasil_pdf_as)

        # --- NSP dan Anuitas ---
        print(f"\n   --- NSP dan Anuitas (Asumsi) ---")
        hasil_nsp_as = mp.nsp_wl_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   6. NSP Whole Life A_{{{usia_x}}}:", hasil_nsp_as)
        hasil_pv_ann_as = mp.pv_annuity_due_wl_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   7. PV Anuitas Whole Life Due ä_{{{usia_x}}}:", hasil_pv_ann_as)

        # --- Momen Curtate ---
        print(f"\n   --- Momen Curtate (K_x) (Asumsi) ---")
        ex_wl_as = mp.ex_curtate_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   a. Ekspektasi (e_{{{usia_x}}}, WL):", ex_wl_as)
        ex_temp_as = mp.ex_curtate_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params, n_temp=n_temporary) # type: ignore
        tampilkan_hasil(f"   b. Ekspektasi (e_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temp):", ex_temp_as)
        e_sq_wl_as = mp.e_sq_curtate_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   c. Momen Kedua (E[K_{{{usia_x}}}^2], WL):", e_sq_wl_as)
        var_k_wl_as = mp.var_k_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   d. Variansi (Var[K_{{{usia_x}}}], WL):", var_k_wl_as)

        # --- Momen Complete ---
        print(f"\n   --- Momen Complete (T_x) (Asumsi) ---")
        ex_c_wl_as = mp.ex_complete_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   e. Ekspektasi (e_circ_{{{usia_x}}}, WL):", ex_c_wl_as)
        ex_c_temp_as = mp.ex_complete_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params, n_temp=n_temporary) # type: ignore
        tampilkan_hasil(f"   f. Ekspektasi (e_circ_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temp):", ex_c_temp_as)
        e_sq_c_wl_as = mp.e_sq_complete_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   g. Momen Kedua (E[T_{{{usia_x}}}^2], WL):", e_sq_c_wl_as)
        var_t_wl_as = mp.var_t_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil(f"   h. Variansi (Var[T_{{{usia_x}}}], WL):", var_t_wl_as)

    except Exception as e_assume:
        print(f"   [ERROR] Terjadi kesalahan pada asumsi {assumption_type}: {e_assume}")
        # import traceback
        # traceback.print_exc()


print("\n" + "=" * 70)
print("              DEMO LENGKAP MORTAPY SELESAI")
print("=" * 70)

        DEMO LENGKAP MORTAPY (TERMASUK MOMEN & DASAR SELEKSI)

--- Parameter Umum yang Digunakan dalam Demo ---
Usia awal (x)         : 65
Suku bunga (i)        : 5.00%
Gender (untuk tabel)  : Wanita
Periode temporary (n) : 10 tahun
Periode probabilitas (n): 5 tahun
Periode tunda (m)       : 3 tahun
Periode kematian (u)    : 7 tahun
Offset waktu (s)        : 0.5 tahun
------------------------------------------------------------

 BAGIAN 1: PERHITUNGAN BERBASIS TABEL MORTALITA (TMI DEFAULT)
 (Fitur Seleksi & Ultima belum didemokan dengan tabel khusus, menggunakan ultima)

Berhasil memuat tabel default: <MortalityTable ultimate='tabel_mortalita_penduduk_indonesia_2023.csv'>

--- Menggunakan parameter: Usia = 65, Gender = wanita, n_temporary = 10 ---

--- 1.1 Fungsi Probabilitas Dasar (Tabel) ---
a. Probabilitas Hidup _{5}p_{65}:


<IPython.core.display.Math object>

Deskripsi: Probabilitas Hidup 5 Tahun (Tabel), Usia Awal 65, Gender Wanita
------------------------------------------------------------
b. Probabilitas Kematian _{5}q_{65}:


<IPython.core.display.Math object>

Deskripsi: Probabilitas Kematian 5 Tahun (Tabel), Usia Awal 65, Gender Wanita
------------------------------------------------------------
c. Prob. Kematian Ditunda _{3}|_{7}q_{65}:


...

<IPython.core.display.Math object>

Deskripsi: Momen Kedua Complete Future Lifetime (Tabel, Frac: UDD), Usia 65, Gender Wanita
------------------------------------------------------------
d. Variansi Complete (Var[T_{65}], WL):


<IPython.core.display.Math object>

Deskripsi: Variansi Complete Future Lifetime (Tabel, Frac: UDD), Usia 65, Gender Wanita
------------------------------------------------------------

 BAGIAN 2: MOMEN BERBASIS ASUMSI DISTRIBUSI MURNI

--- 2.1 Menggunakan Asumsi: q_x konstan = 0.03 ---


NameError: name 'periode_t_float' is not defined